# Multi-agent routing queries

**Notebook 5 of 5.** Demonstrates end-to-end multi-agent routing: each user question is
classified by the orchestrator, routed to the correct domain specialist, and answered
from that specialist's Contoso knowledge base.

```
User query
     │
     ▼
 orchestrator  →  classifies: HR | MARKETING | PRODUCTS
     │
     ▼
 ┌───┼──────────┐
 │             │
hr-agent  marketing-agent  products-agent
 │             │                 │
 ▼             ▼                 ▼
contoso-kb-hr  contoso-kb-marketing  contoso-kb-products
```

## Prerequisites

1. **Run `11-04-multi-agent-setup.ipynb`** - all agents and KBs must be validated.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - run `az login` before executing cells.

## Imports and configuration

In [1]:
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

repo_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent
load_dotenv(repo_root / '.env', override=True)

# Make local agents module importable
lab_dir = Path.cwd()
if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))

PROJECT_ENDPOINT = os.environ['CONTOSO_FOUNDRY_PROJECT_ENDPOINT']
SEARCH_ENDPOINT  = os.environ['CONTOSO_SEARCH_ENDPOINT']
CHAT_MODEL       = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')
APIM_CONNECTION  = os.environ['CONTOSO_APIM_CONNECTION']

credential = DefaultAzureCredential()

print(f'Project endpoint : {PROJECT_ENDPOINT}')
print(f'Search endpoint  : {SEARCH_ENDPOINT}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'APIM connection  : {APIM_CONNECTION}')

Project endpoint : https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/contoso-project
Search endpoint  : https://contoso-search-n5d3ja.search.windows.net
Chat model       : gpt-4.1-mini
APIM connection  : contoso-apim-connection


## Instantiate agents and workflow

In [2]:
from agents import (
    create_hr_agent,
    create_marketing_agent,
    create_products_agent,
    create_orchestrator_agent,
    build_contoso_workflow,
)

hr_agent        = create_hr_agent(PROJECT_ENDPOINT, SEARCH_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)
marketing_agent = create_marketing_agent(PROJECT_ENDPOINT, SEARCH_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)
products_agent  = create_products_agent(PROJECT_ENDPOINT, SEARCH_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)
orchestrator    = create_orchestrator_agent(PROJECT_ENDPOINT, CHAT_MODEL, APIM_CONNECTION, credential)

workflow = build_contoso_workflow(
    orchestrator=orchestrator,
    hr_agent=hr_agent,
    marketing_agent=marketing_agent,
    products_agent=products_agent,
)

print('Agents ready:')
for a in [orchestrator, hr_agent, marketing_agent, products_agent]:
    print(f'  {a.name}')
print('Workflow built.')

Agents ready:
  contoso-orchestrator
  contoso-hr-agent
  contoso-marketing-agent
  contoso-products-agent
Workflow built.


## Define ask() helper

The helper performs two steps:
1. **Classify** - runs the orchestrator to determine which specialist to use.
2. **Answer** - runs the specialist directly against its knowledge base.

Exponential backoff handles transient 429 rate-limit errors (10 s, 20 s, 40 s, 80 s).

In [3]:
async def ask(question: str) -> tuple[str, str]:
    """Route `question` through the multi-agent system.

    Returns:
        (agent_name, response_text) where agent_name is the specialist that answered.

    The two-step implementation makes the routing decision explicit:
      Step 1 - orchestrator classifies the query (HR | MARKETING | PRODUCTS)
      Step 2 - the correct specialist agent answers from its knowledge base

    Retries with exponential backoff on RateLimitError (up to 4 retries).
    """
    for attempt in range(5):
        try:
            # Step 1: classify
            classification = await orchestrator.run(question)
            label = classification.text.strip().upper()

            # Step 2: select specialist
            if 'HR' in label and 'MARKETING' not in label:
                specialist = hr_agent
            elif 'MARKETING' in label:
                specialist = marketing_agent
            else:
                specialist = products_agent

            # Step 3: answer from knowledge base
            result = await specialist.run(question)
            return (specialist.name, result.text)

        except Exception as e:
            if 'rate' in str(e).lower() or '429' in str(e):
                if attempt == 4:
                    raise
                wait = 10 * (2 ** attempt)   # 10s, 20s, 40s, 80s
                print(f'Rate limited - waiting {wait}s before retry {attempt + 1}/4 ...')
                time.sleep(wait)
            else:
                raise


print('ask() helper ready.')

ask() helper ready.


---
## Query 1: HR domain

**Expected routing:** `contoso-hr-agent` → `contoso-kb-hr`

In [4]:
from display_helpers import show_routing_decision, show_agent_response

query = "What is Contoso's remote work policy?"
agent_name, response_text = await ask(query)

show_routing_decision(query, agent_name)
show_agent_response(query, response_text, agent_name)

Contoso's Remote Work Policy permits employees to work from approved locations outside the primary office with manager approval and based on role eligibility. Employees must maintain a secure, distraction-free workspace with reliable internet of at least 25 Mbps. Core working hours are 10:00 AM to 3:00 PM local time for synchronous collaboration. Remote workers must attend all scheduled team meetings via video and keep their calendars updated.

Equipment like laptops and peripherals is provided by Contoso IT, with employees responsible for care and return upon separation. Personal use of company equipment is prohibited. Data security requirements apply equally to remote and in-office employees, including mandatory VPN use for internal systems.

Employees can work remotely up to five days per week or on a hybrid basis with approval. New employees must be on-site for their first 30 days for onboarding. Remote work privileges may be revoked if performance, collaboration, or security standards are not met. Requests to work remotely from international locations require prior HR and Legal approval due to tax and compliance reasons. The policy is reviewed annually and updated as needed.

---
## Query 2: Marketing domain

**Expected routing:** `contoso-marketing-agent` → `contoso-kb-marketing`

In [5]:
query = 'What are the key elements of the Contoso brand guidelines?'
agent_name, response_text = await ask(query)

show_routing_decision(query, agent_name)
show_agent_response(query, response_text, agent_name)

The key elements of the Contoso brand guidelines include the Contoso Blue primary logo with strict usage rules, a secondary color palette featuring Contoso Blue, Teal, Grey, White, and accent colors Orange and Green for specific uses, Segoe UI typography for headings and body, a natural and warm photography style focusing on people, Fluent UI iconography at a 24px baseline, mandatory brand team review for new materials, approved templates on the Brand Hub, and a contact email for questions. These details are from the "Contoso Brand Guidelines" document.

---
## Query 3: Products domain

**Expected routing:** `contoso-products-agent` → `contoso-kb-products`

In [6]:
query = 'What are the specifications of the ContosoBook Pro?'
agent_name, response_text = await ask(query)

show_routing_decision(query, agent_name)
show_agent_response(query, response_text, agent_name)

The ContosoBook Pro specifications are:

- Processor: Intel Core Ultra 7 Series 2 with Contoso ContextSense NPU, up to 48 TOPS AI compute.
- Display: 14-inch 2880×1800 OLED, 120Hz refresh rate, 400 nits brightness, 100% DCI-P3 color.
- Battery Life: Up to 18 hours video playback, 80W Thunderbolt 4 charging.
- Memory: 16GB or 32GB LPDDR5x.
- Storage: 512GB, 1TB, or 2TB NVMe PCIe Gen 4.
- Ports: Two Thunderbolt 4, two USB-A 3.2 Gen 2, HDMI 2.1, SD card reader, 3.5mm combo audio jack.
- Connectivity: Wi-Fi 7 and Bluetooth 5.4.
- Build: Aircraft-grade aluminum chassis, weight 1.28 kg.
- Keyboard: Full-size island layout, 1.5mm key travel, backlit keys, fingerprint reader in power button, IR camera for Windows Hello.
- Dimensions: 311mm × 222mm × 14.8mm.
- Colors: Platinum Silver and Midnight Black.
- OS: Windows 11 Pro with one-year Contoso Premier Support.
- Warranty: Three-year on-site warranty available.
- SKU: CBP-14-001.
- Starting Price: $1,299 USD.

---
## Query 4: Ambiguous (HR default)

Performance evaluation could be classified as HR or Products depending on phrasing.
The orchestrator should classify this as HR given the Contoso HR KB covers performance reviews.

**Expected routing:** `contoso-hr-agent` → `contoso-kb-hr`

In [7]:
query = 'How does performance evaluation work at Contoso?'
agent_name, response_text = await ask(query)

show_routing_decision(query, agent_name)
show_agent_response(query, response_text, agent_name)

At Contoso, the performance evaluation process happens twice a year: a mid-year check-in in July and an annual review in December. Employees start by completing a self-assessment in Workday, then managers evaluate them, calibrate ratings within their teams, and participate in a cross-functional calibration led by HR to ensure consistent ratings. Ratings use a five-point scale from Exceptional to Unsatisfactory. Unsatisfactory ratings trigger a Performance Improvement Plan with 60-day check-ins. Merit increases are based on the annual rating and take effect January 1. The mid-year check-in focuses on development and goal progress but doesn't affect compensation. Senior Managers and above also receive 360-degree feedback, and continuous feedback is encouraged year-round via Workday. Employees can discuss their review with their manager and HR if needed. All performance records are kept for three years. (Performance Review Process)

---
## Query 5: Out-of-scope

A general knowledge question that does not appear in any Contoso knowledge base.
The specialist agent should respond that this information is not available in its
knowledge base rather than hallucinating an answer.

**Expected routing:** products-agent (Default case) - returns a KB-grounded non-answer

In [8]:
query = 'What is the capital of France?'
agent_name, response_text = await ask(query)

show_routing_decision(query, agent_name)
show_agent_response(query, response_text, agent_name)

The capital of France is Paris.

---
## Summary

| Query | Expected agent | Domain |
|-------|---------------|--------|
| Remote work policy | `contoso-hr-agent` | HR |
| Brand guidelines | `contoso-marketing-agent` | Marketing |
| ContosoBook Pro specs | `contoso-products-agent` | Products |
| Performance evaluation | `contoso-hr-agent` | HR (ambiguous) |
| Capital of France | `contoso-products-agent` | Out-of-scope (Default) |

**Next steps:** The lab infrastructure is complete. Use `11-04-multi-agent-setup.ipynb` to
add further specialist agents or extend the knowledge bases with additional Contoso documents.